# Functions & Libraries

In [0]:
#Call libraries
from pyspark.sql import functions as F
import pandas as pd

## Baseline functions

In [0]:
#Funtion to return valid candidates for an empty Sudoku cell
def get_cell_candidates(grid, row, col):
    """ Return valid candidates for an empty Sudoku cell. """

    if grid[row][col] != 0:
        return []

    row_values = set(grid[row])
    column_values = {grid[r][col] for r in range(9)}

    box_row = (row // 3) * 3
    box_col = (col // 3) * 3

    box_values = {
        grid[r][c]
        for r in range(box_row, box_row + 3)
        for c in range(box_col, box_col + 3)
    }

    used_numbers = row_values | column_values | box_values

    return sorted(set(range(1, 10)) - used_numbers)

#Funtion to apply the Naked Single technique
def solve_naked_singles(grid):
    """ Apply Naked Single moves until no further progress is possible. """

    board = [row.copy() for row in grid]
    naked_single_steps = 0

    while True:
        progress = False

        for row in range(9):
            for col in range(9):
                if board[row][col] == 0:
                    candidates = get_cell_candidates(board, row, col)

                    if len(candidates) == 1: #If there is exactly one candidate, it fills the cell and counts the operation
                        board[row][col] = candidates[0]
                        naked_single_steps += 1
                        progress = True

        if not progress:
            break

    solved = all(
        board[row][col] != 0
        for row in range(9)
        for col in range(9)
    )

    return {
        "board": board,
        "naked_single_steps": naked_single_steps,
        "solved": solved,
        "remaining_empty_cells": sum(
            cell == 0
            for row in board
            for cell in row
        )
    }

    #Notes:
        #If solved = True, it means the Sudoku has been completely solved using this technique
        #If solved = False, it means we will need other strategies to continue the solution process

#function to apply hidden single technique
def find_hidden_singles(grid):
    """ Find cells that contain the only possible position for a number within a row, column, or 3x3 box. """

    hidden_singles = {}

    units = []

    # Rows
    for row in range(9):
        units.append(
            [(row, col) for col in range(9)]
        )

    # Columns
    for col in range(9):
        units.append(
            [(row, col) for row in range(9)]
        )

    # 3x3 boxes
    for box_row in range(0, 9, 3):
        for box_col in range(0, 9, 3):
            units.append([
                (row, col)
                for row in range(box_row, box_row + 3)
                for col in range(box_col, box_col + 3)
            ])

    for unit in units:
        for number in range(1, 10):
            possible_cells = []

            for row, col in unit:
                if grid[row][col] == 0:
                    candidates = get_cell_candidates(grid, row, col)

                    if number in candidates:
                        possible_cells.append((row, col))

            if len(possible_cells) == 1:
                row, col = possible_cells[0]

                if (row, col) not in hidden_singles:
                    hidden_singles[(row, col)] = number

    return hidden_singles

#Funtion to apply the Hidden Single technique with the Naked Single technique
def solve_with_singles(grid):
    """ Solve a Sudoku using Naked Singles and Hidden Singles. """

    board = [row.copy() for row in grid]

    naked_single_steps = 0
    hidden_single_steps = 0

    while True:
        progress = False

        #Apply Naked Singles first
        for row in range(9):
            for col in range(9):
                if board[row][col] == 0:
                    candidates = get_cell_candidates(board, row, col)

                    if len(candidates) == 1:
                        board[row][col] = candidates[0]
                        naked_single_steps += 1
                        progress = True

        if progress:
            continue

        #Apply one Hidden Single, then recalculate all candidates
        hidden_singles = find_hidden_singles(board)

        for (row, col), number in hidden_singles.items():
            if board[row][col] == 0:
                candidates = get_cell_candidates(board, row, col)

                if number in candidates:
                    board[row][col] = number
                    hidden_single_steps += 1
                    progress = True
                    break

        if not progress:
            break

    remaining_empty_cells = sum(
        cell == 0
        for row in board
        for cell in row
    )

    return {
        "board": board,
        "naked_single_steps": naked_single_steps,
        "hidden_single_steps": hidden_single_steps,
        "solved": remaining_empty_cells == 0,
        "remaining_empty_cells": remaining_empty_cells,
    }

#Funtion to apply the Naked Single and Hidden Single techniques with the number of rounds 
def solve_with_singles_and_rounds(grid):
    """ Solve a Sudoku and track logical solving rounds. """

    board = [row.copy() for row in grid]

    naked_single_steps = 0
    hidden_single_steps = 0
    solver_rounds = 0

    while True:
        round_progress = False

        #Apply Naked Singles until no more are available
        while True:
            naked_progress = False

            for row in range(9):
                for col in range(9):
                    if board[row][col] == 0:
                        candidates = get_cell_candidates(board, row, col)

                        if len(candidates) == 1:
                            board[row][col] = candidates[0]
                            naked_single_steps += 1
                            naked_progress = True
                            round_progress = True

            if not naked_progress:
                break

        #Apply one Hidden Single at a time
        hidden_singles = find_hidden_singles(board)

        for (row, col), number in hidden_singles.items():
            if board[row][col] == 0:
                candidates = get_cell_candidates(board, row, col)

                if number in candidates:
                    board[row][col] = number
                    hidden_single_steps += 1
                    round_progress = True
                    break

        if not round_progress:
            break

        solver_rounds += 1

    remaining_empty_cells = sum(
        cell == 0
        for row in board
        for cell in row
    )

    return {
        "board": board,
        "naked_single_steps": naked_single_steps,
        "hidden_single_steps": hidden_single_steps,
        "solver_rounds": solver_rounds,
        "solved": remaining_empty_cells == 0,
        "remaining_empty_cells": remaining_empty_cells,
    }

#Function to apply the Backtracking and MRV techniques
def solve_with_backtracking(grid):
    """ Solve a Sudoku using deterministic backtracking and MRV. """

    board = [row.copy() for row in grid]

    search_nodes = 0
    backtracks = 0
    max_search_depth = 0

    def search(depth=0):
        nonlocal search_nodes, backtracks, max_search_depth

        search_nodes += 1
        max_search_depth = max(max_search_depth, depth)

        best_cell = None
        best_candidates = None

        # Select the empty cell with the fewest valid candidates.
        for row in range(9):
            for col in range(9):
                if board[row][col] != 0:
                    continue

                candidates = get_cell_candidates(board, row, col)

                if not candidates:
                    backtracks += 1
                    return False

                if (
                    best_candidates is None
                    or len(candidates) < len(best_candidates)
                ):
                    best_cell = (row, col)
                    best_candidates = candidates

                    if len(best_candidates) == 1:
                        break

            if best_candidates is not None and len(best_candidates) == 1:
                break

        # No empty cells remain.
        if best_cell is None:
            return True

        row, col = best_cell

        for number in best_candidates:
            board[row][col] = number

            if search(depth + 1):
                return True

            board[row][col] = 0
            backtracks += 1

        return False

    solved = search()

    return {
        "board": board,
        "solved": solved,
        "search_nodes": search_nodes,
        "backtracks": backtracks,
        "max_search_depth": max_search_depth,
    }

# Problem-solving technique

## Baseline metrics

In [0]:
#Solvers:
    #naked_single_steps -> Entries made using the "Naked Single" technique -> A cell where only one candidate is possible
    #hidden_single_steps -> Entries made using the "Hidden Single" technique -> A number that can only occupy one position within a given unit
    #backtracking_nodes_MRV -> The number of nodes explored during the search with MRV -> Exploratory attempts and subsequent backtracks, used when logical techniques are insufficient to solve the puzzle
    #solver_iterations -> Cycles required to apply the strategies

### Naked Single 

In [0]:
#Naked Single technique test one case
gold_df = spark.table("workspace.sudoku_gold.game_features")
sample_puzzle = gold_df.select("quizzes").first()["quizzes"]

grid = [
    [int(sample_puzzle[row * 9 + col]) for col in range(9)]
    for row in range(9)
]

result = solve_naked_singles(grid)

print(f"Naked Single steps: {result['naked_single_steps']}") 
print(f"Remaining empty cells: {result['remaining_empty_cells']}")
print(f"Solved: {result['solved']}")


In [0]:
#Retrieve the reference solution for the same puzzle
sample_record = (
    spark.table("workspace.sudoku_bronze.raw_games")
    .filter(F.col("quizzes") == sample_puzzle)
    .select("solutions")
    .first()
)

reference_solution = sample_record["solutions"]

#Convert the reference solution into a 9x9 grid
reference_grid = [
    [int(reference_solution[row * 9 + col]) for col in range(9)]
    for row in range(9)
]

#Compare the solver output with the reference solution
solution_matches = result["board"] == reference_grid

print(f"Solver completed: {result['solved']}")
print(f"Solution matches reference: {solution_matches}")
print(f"Naked Single steps: {result['naked_single_steps']}")

In [0]:
#Naked Single technique test one hundred case
sample_df = (
    spark.table("workspace.sudoku_bronze.raw_games")
    .select("quizzes", "solutions")
    .limit(100)
    .collect()
)

test_results = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_naked_singles(grid)

    test_results.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "naked_single_steps": result["naked_single_steps"],
        "remaining_empty_cells": result["remaining_empty_cells"],
    })

total_puzzles = len(test_results)
solved_count = sum(result["solved"] for result in test_results)
matching_count = sum(result["solution_matches"] for result in test_results)

print(f"Puzzles tested: {total_puzzles}")
print(f"Solved by Naked Singles: {solved_count}")
print(f"Solutions matching reference: {matching_count}")
print(f"Solve rate: {solved_count / total_puzzles:.2%}")

In [0]:
#Naked Single technique test one thousand random case
sample_df = (
    spark.table("workspace.sudoku_bronze.raw_games")
    .select("quizzes", "solutions")
    .orderBy(F.rand(seed=42))
    .limit(1000)
    .collect()
)

test_results = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_naked_singles(grid)

    test_results.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "naked_single_steps": result["naked_single_steps"],
        "remaining_empty_cells": result["remaining_empty_cells"],
    })

total_puzzles = len(test_results)
solved_count = sum(item["solved"] for item in test_results)
matching_count = sum(item["solution_matches"] for item in test_results)

print(f"Puzzles tested: {total_puzzles}")
print(f"Solved by Naked Singles: {solved_count}")
print(f"Solutions matching reference: {matching_count}")
print(f"Solve rate: {solved_count / total_puzzles:.2%}")

In [0]:
#Analyze the number of candidates and the solved cells
results_df = pd.DataFrame(test_results)

print("Naked Single steps — descriptive statistics")
print(results_df["naked_single_steps"].describe())

print("\nNaked Single steps — frequency distribution")
print(
    results_df["naked_single_steps"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nRemaining empty cells — frequency distribution")
print(
    results_df["remaining_empty_cells"]
    .value_counts()
    .sort_index()
    .to_string()
)

In [0]:
#Notes:
    #The solver is working, but we need to be careful with the interpretation
    #The puzzles contain between 45 and 50 empty cells that are solved by the algorithm
    #Since all of them were solved, the `naked_single_steps` count matches the number of empty cells in these cases
    #Therefore, this variable essentially replicates information we already have in the Gold layer: `empty_cells = 81 - given_cells`

In [0]:
#Naked Single technique test ten thousand case
sample_df = (
    spark.table("workspace.sudoku_bronze.raw_games")
    .select("quizzes", "solutions")
    .sample(withReplacement=False, fraction=0.02, seed=42)
    .limit(10_000)
    .collect()
)

test_results = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_naked_singles(grid)

    test_results.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "naked_single_steps": result["naked_single_steps"],
        "remaining_empty_cells": result["remaining_empty_cells"],
        "quizzes": puzzle,
        "reference_solution": reference_solution
    })

total_puzzles = len(test_results)
solved_count = sum(item["solved"] for item in test_results)
matching_count = sum(item["solution_matches"] for item in test_results)

print(f"Puzzles tested: {total_puzzles}")
print(f"Solved by Naked Singles: {solved_count}")
print(f"Solutions matching reference: {matching_count}")
print(f"Solve rate: {solved_count / total_puzzles:.2%}")

In [0]:
#Investigate the 4 unsolved puzzles
unsolved_results = [
    item for item in test_results
    if not item["solved"]
]

print(f"Unsolved puzzles: {len(unsolved_results)}")

for index, item in enumerate(unsolved_results, start=1):
    print(
        f"Puzzle {index}: "
        f"Naked Single steps = {item['naked_single_steps']}, "
        f"Remaining empty cells = {item['remaining_empty_cells']}"
    )

### Hidden Single 

In [0]:
#Discover the cells not solved by Naked Singles
first_unsolved = next(
    item for item in test_results
    if not item["solved"]
)

puzzle = first_unsolved["quizzes"]

grid = [
    [int(puzzle[row * 9 + col]) for col in range(9)]
    for row in range(9)
]

hidden_singles = find_hidden_singles(grid)

print(f"Hidden Singles found: {len(hidden_singles)}")
print(hidden_singles)

### Hidden Single + Naked Single

In [0]:
#Test the combined solver on the puzzle that got stuck
result = solve_with_singles(grid)

print(f"Solved: {result['solved']}")
print(f"Naked Single steps: {result['naked_single_steps']}")
print(f"Hidden Single steps: {result['hidden_single_steps']}")
print(f"Remaining empty cells: {result['remaining_empty_cells']}")

In [0]:
#Test the one thousand puzzles using combined solver
test_results_combined = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_with_singles(grid)

    test_results_combined.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "naked_single_steps": result["naked_single_steps"],
        "hidden_single_steps": result["hidden_single_steps"],
        "remaining_empty_cells": result["remaining_empty_cells"],
    })

total_puzzles = len(test_results_combined)
solved_count = sum(item["solved"] for item in test_results_combined)
matching_count = sum(
    item["solution_matches"] for item in test_results_combined
)

print(f"Puzzles tested: {total_puzzles}")
print(f"Solved by combined strategies: {solved_count}")
print(f"Solutions matching reference: {matching_count}")
print(f"Solve rate: {solved_count / total_puzzles:.2%}")

In [0]:
#Analyze how many entries relied on each strategy and how many puzzles required Hidden Singles.
combined_df = pd.DataFrame(test_results_combined)

print("Naked Single steps")
print(combined_df["naked_single_steps"].describe())

print("\nHidden Single steps")
print(combined_df["hidden_single_steps"].describe())

print("\nPuzzles requiring Hidden Singles")
print(
    (combined_df["hidden_single_steps"] > 0).sum()
)

print("\nHidden Single steps distribution")
print(
    combined_df["hidden_single_steps"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nSolution validation")
print(
    f"All solved: {combined_df['solved'].all()}"
)
print(
    f"All solutions match reference: "
    f"{combined_df['solution_matches'].all()}"
)

In [0]:
#Notes:
    #The results reveal something very important about the dataset: Hidden Singles were required in only 4 out of the 10,000 puzzles (0.04%). The other 9,996 were solved exclusively using Naked Singles

#Insights:
    #The Hidden Singles strategy is not a common technique for solving Sudoku puzzles. It is a more advanced technique that is typically used when other strategies have been exhausted
    #Measure the resolution rounds in the combined solver


In [0]:
#Test the combined solver with round counter
record = sample_df[0]
puzzle = record["quizzes"]

grid = [
    [int(puzzle[row * 9 + col]) for col in range(9)]
    for row in range(9)
]

result = solve_with_singles_and_rounds(grid)

print(f"Solved: {result['solved']}")
print(f"Naked Single steps: {result['naked_single_steps']}")
print(f"Hidden Single steps: {result['hidden_single_steps']}")
print(f"Solver rounds: {result['solver_rounds']}")
print(f"Remaining empty cells: {result['remaining_empty_cells']}")


In [0]:

#Run the solver on the full sample
round_results = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_with_singles_and_rounds(grid)

    round_results.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "naked_single_steps": result["naked_single_steps"],
        "hidden_single_steps": result["hidden_single_steps"],
        "solver_rounds": result["solver_rounds"],
        "remaining_empty_cells": result["remaining_empty_cells"],
    })

round_df = pd.DataFrame(round_results)

print("Solver rounds — descriptive statistics")
print(round_df["solver_rounds"].describe())

print("\nSolver rounds — frequency distribution")
print(
    round_df["solver_rounds"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nValidation")
print(f"All solved: {round_df['solved'].all()}")
print(f"All solutions match reference: {round_df['solution_matches'].all()}")


In [0]:
#Evaluate the diversity of the Gold features
sample_puzzles = [record["quizzes"] for record in sample_df]

gold_sample_df = (
    spark.table("workspace.sudoku_gold.game_features")
    .filter(F.col("quizzes").isin(sample_puzzles))
    .select(
        "given_cells",
        "row_clue_std",
        "column_clue_std",
        "box_clue_std",
        "avg_candidate_count",
        "single_candidate_cells",
        "max_candidate_count",
    )
)

gold_sample_pd = gold_sample_df.toPandas()

print("Gold feature summary")
print(gold_sample_pd.describe().round(4))

print("\nGiven cells distribution")
print(
    gold_sample_pd["given_cells"]
    .value_counts()
    .sort_index()
    .to_string()
)

In [0]:
#Notes:
    #The distribution of `given_cells` is highly concentrated: 4,465 puzzles have 34 numbers filled in at the start, and 9,181 have between 33 and 35. This helps explain why the solver's behavior is so uniform across this sample, though it does not yet prove that the number of clues is the sole cause

    #We also observe variation in the "Gold" features—specifically, `avg_candidate_count`, `single_candidate_cells`, and the standard deviations of the clues vary from one puzzle to another. Therefore, while we have structural data to investigate, we still require a target variable that represents something beyond these features themselves


In [0]:
#Insights:
    #Generate an algorithmic complexity target
    #Implement a solver using deterministic search, selecting first the empty cell with the fewest candidates (MRV — Minimum Remaining Values)
    #Count `search_nodes`, `backtracks`, and `max_search_depth`
    #Validate the solutions against the dataset's ground truth
    #Analyze the distribution of these metrics across the same 10,000 puzzles
    #Only then, decide whether they can serve as a target for regression and how to define difficulty classes

### Backtracking and MRV

In [0]:
#Test the backtracking solver
record = sample_df[0]
puzzle = record["quizzes"]
reference_solution = record["solutions"]

grid = [
    [int(puzzle[row * 9 + col]) for col in range(9)]
    for row in range(9)
]

result = solve_with_backtracking(grid)

reference_grid = [
    [
        int(reference_solution[row * 9 + col])
        for col in range(9)
    ]
    for row in range(9)
]

print(f"Solved: {result['solved']}")
print(f"Solution matches reference: {result['board'] == reference_grid}")
print(f"Search nodes: {result['search_nodes']}")
print(f"Backtracks: {result['backtracks']}")
print(f"Maximum search depth: {result['max_search_depth']}")

In [0]:
#Notes:
    #Even though backtracking was used, this puzzle was solved without exploring alternatives

In [0]:
#Check whether the metrics vary across the 10,000 puzzles
backtracking_results = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_with_backtracking(grid)

    backtracking_results.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "search_nodes": result["search_nodes"],
        "backtracks": result["backtracks"],
        "max_search_depth": result["max_search_depth"],
    })

backtracking_df = pd.DataFrame(backtracking_results)

print("Search nodes — descriptive statistics")
print(backtracking_df["search_nodes"].describe())

print("\nBacktracks — descriptive statistics")
print(backtracking_df["backtracks"].describe())

print("\nMaximum search depth — descriptive statistics")
print(backtracking_df["max_search_depth"].describe())

print("\nValidation")
print(f"All solved: {backtracking_df['solved'].all()}")
print(
    "All solutions match reference: "
    f"{backtracking_df['solution_matches'].all()}"
)

print("\nBacktracks frequency")
print(
    backtracking_df["backtracks"]
    .value_counts()
    .sort_index()
    .to_string()
)

In [0]:
#Notes:
    #9,998 puzzles required no backtracking
    #Only 2 required backtracking: one with 5 backtracks and the other with 15 backtracks
    #The number of `search_nodes` ranged from 46 to 63, though this variation is strongly linked to the number of cells filled during the search
    #The maximum depth also closely tracks the number of empty cells

In [0]:
#Identify puzzles that required backtracking
exceptional_indices = backtracking_df.index[
    backtracking_df["backtracks"] > 0
].tolist()

print(f"Puzzles requiring backtracking: {len(exceptional_indices)}")

exceptional_puzzles = []

for index in exceptional_indices:
    record = sample_df[index]
    metrics = backtracking_df.loc[index]

    exceptional_puzzles.append({
        "sample_index": index,
        "quizzes": record["quizzes"],
        "solutions": record["solutions"],
        "search_nodes": int(metrics["search_nodes"]),
        "backtracks": int(metrics["backtracks"]),
        "max_search_depth": int(metrics["max_search_depth"]),
    })

exceptional_quizzes = [
    item["quizzes"] for item in exceptional_puzzles
]

gold_exceptional_df = (
    spark.table("workspace.sudoku_gold.game_features")
    .filter(F.col("quizzes").isin(exceptional_quizzes))
    .select(
        "quizzes",
        "given_cells",
        "row_clue_std",
        "column_clue_std",
        "box_clue_std",
        "avg_candidate_count",
        "single_candidate_cells",
        "max_candidate_count",
    )
    .toPandas()
)

exceptional_df = pd.DataFrame(exceptional_puzzles).merge(
    gold_exceptional_df,
    on="quizzes",
    how="left",
)

print("\nExceptional puzzles and their metrics")
print(
    exceptional_df.drop(columns=["quizzes", "solutions"])
    .to_string(index=False)
)

In [0]:
#Notes:
    #Puzzle 1725 required more attempts and backtracking than puzzle 5357, even though both began with 33 pre-filled numbers
    #This already reveals something interesting: the initial number of clues, on its own, does not fully account for the effort required to solve a puzzle
    #However, these are just two exceptional cases within a sample of 10,000 puzzles. We cannot draw general conclusions about difficulty based on them

In [0]:
#Evaluate the relationship between features and search effort
analysis_df = gold_sample_pd.reset_index(drop=True).copy()

analysis_df["search_nodes"] = backtracking_df["search_nodes"].to_numpy()
analysis_df["backtracks"] = backtracking_df["backtracks"].to_numpy()
analysis_df["max_search_depth"] = (
    backtracking_df["max_search_depth"].to_numpy()
)

columns_to_analyze = [
    "given_cells",
    "row_clue_std",
    "column_clue_std",
    "box_clue_std",
    "avg_candidate_count",
    "single_candidate_cells",
    "max_candidate_count",
    "search_nodes",
    "backtracks",
    "max_search_depth",
]

print(
    analysis_df[columns_to_analyze]
    .corr(method="spearman")
    .round(4)
    .to_string()
)

#Notes:
    #Spearman's correlation helps identify monotonic relationships without requiring them to be linear. Since nearly all backtrack values ​​are zero, the correlation should be interpreted with caution (~+1 -> perfect correlation, ~-1 -> perfect anti-correlation, ~0 -> no correlation)
    #Why use Spearman instead of Pearson?
        #Pearson measures the strength of a linear relationship between the values ​​of variables
        #Spearman assesses the relationship between their relative positions (ranks), allowing for the identification of monotonic relationships that are not linear, and is less sensitive to extreme values ​​in certain situations

In [0]:
#Notes:
    #given_cells × avg_candidate_count -> -0.532 -> Moderate negative association
    #avg_candidate_count × single_candidate_cells -> -0.636 -> Strong negative association
    #avg_candidate_count × search_nodes -> +0.532 -> Moderate positive association
    #given_cells × search_nodes	-> -0.999 -> Result that requires verification

In [0]:
#Associate solver metrics with their original puzzles
backtracking_df["quizzes"] = [
    record["quizzes"] for record in sample_df
]

#Retrieve Gold features for the same puzzles
sample_puzzles = backtracking_df["quizzes"].tolist()

gold_sample_pd = (
    spark.table("workspace.sudoku_gold.game_features")
    .filter(F.col("quizzes").isin(sample_puzzles))
    .select(
        "quizzes",
        "given_cells",
        "row_clue_std",
        "column_clue_std",
        "box_clue_std",
        "avg_candidate_count",
        "single_candidate_cells",
        "max_candidate_count",
    )
    .toPandas()
)

#Join by puzzle identity, not by row position
analysis_df = backtracking_df.merge(
    gold_sample_pd,
    on="quizzes",
    how="inner",
    validate="one_to_one",
)

columns_to_analyze = [
    "given_cells",
    "row_clue_std",
    "column_clue_std",
    "box_clue_std",
    "avg_candidate_count",
    "single_candidate_cells",
    "max_candidate_count",
    "search_nodes",
    "backtracks",
    "max_search_depth",
]

print(f"Matched puzzles: {len(analysis_df)}")

print(
    analysis_df[columns_to_analyze]
    .corr(method="spearman")
    .round(4)
    .to_string()
)

In [0]:
#Notes:
    #given_cells × search_nodes -> -0.9999 -> More clues, fewer search nodes
    #given_cells × max_search_depth	-> -1.0000 -> More clues, shallower search depth
    #search_nodes × max_search_depth -> +0.9999 -> More nodes, greater search depth
    #avg_candidate_count × search_nodes	-> +0.5319 -> More candidates, tendency toward more nodes
    #avg_candidate_count × single_candidate_cells -> -0.6356 -> More candidates, tendency toward fewer cells with a single candidate
    #given_cells × backtracks -> -0.0154 -> Virtually no monotonic association

In [0]:
#Insights:
    #Don't use `search_nodes` on its own as a difficulty label for the model. Instead, we need to evaluate a solver that accounts for more sophisticated solving techniques—such as naked pairs, pointing pairs, and other strategies. This way, we’ll have a better chance of generating a metric that shows real variation across the puzzles
    #Advanceds tests:
        #Naked Pairs -> Identify two cells within the same unit that share the exact same two candidates, and eliminate those candidates from all other cells in that unit
        #Pointing Pairs -> Identify when a candidate within a 3×3 block appears only in a single row or column of that block, and eliminate it from the remainder of that row or column
        #Re-evaluate the solver using a sample of 10,000 puzzles, measuring the number of puzzles solved, techniques employed, eliminations made, and backtracking attempts
        #Compare the distribution of these metrics against the current solver to determine whether we have achieved a target with sufficient variation for the model

## Advanced metrics